# ATTENTION MECHANISM

### >Simple self-attention mechanism without trainable weights

In [2]:
import torch

# Format tensors to show 4 decimal places, avoid scientific notation, and look cleaner
torch.set_printoptions(precision=4, sci_mode=False, edgeitems=5)


inputs = torch.tensor(
    [[0.43, 0.15, 0.89],  # Your      (x^1)
     [0.55, 0.87, 0.66],  # journey   (x^2)
     [0.57, 0.85, 0.64],  # starts    (x^3)
     [0.22, 0.58, 0.33],  # with      (x^4)
     [0.77, 0.25, 0.10],  # one       (x^5)
     [0.05, 0.80, 0.55]]  # step      (x^6)
)

In [3]:
input_query=inputs[1]
input_query

tensor([0.5500, 0.8700, 0.6600])

In [4]:
input1=inputs[0]
print(f"dot product of inputs {input_query} and {input1} is:-")
print(torch.dot(input1,input_query))

dot product of inputs tensor([0.5500, 0.8700, 0.6600]) and tensor([0.4300, 0.1500, 0.8900]) is:-
tensor(0.9544)


In [5]:
query=inputs[1] #2nd input token 
attention_score=torch.empty(inputs.shape[0])
for i,x in enumerate(inputs):
    attention_score[i]=torch.dot(x,query) #score of every token based on 2nd input token 
print(attention_score) 

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


## Normalization using softmax 

In [6]:
temp_weights=attention_score / attention_score.sum()
temp_weights

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [7]:
temp_weights.sum()

tensor(1.0000)

In [8]:
def softmax(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)
softmax(attention_score)    

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [9]:
attention_weights=torch.softmax(attention_score,dim=0)

In [10]:
query = inputs[1]  # 2nd input token is the query

context_vector = torch.zeros(query.shape)

for i, x in enumerate(inputs):
    print(f"{attention_weights[i]}---------->{inputs[i]}")
    context_vector += attention_weights[i] * x

print(context_vector) #for input token 2

0.13854756951332092---------->tensor([0.4300, 0.1500, 0.8900])
0.2378913015127182---------->tensor([0.5500, 0.8700, 0.6600])
0.23327402770519257---------->tensor([0.5700, 0.8500, 0.6400])
0.12399158626794815---------->tensor([0.2200, 0.5800, 0.3300])
0.10818186402320862---------->tensor([0.7700, 0.2500, 0.1000])
0.15811361372470856---------->tensor([0.0500, 0.8000, 0.5500])
tensor([0.4419, 0.6515, 0.5683])


### for all tokens

In [11]:
attn_score=torch.empty(6,6)
for i,x in enumerate(inputs):
    for j,y in enumerate(inputs):
        attn_score[i,j]=torch.dot(x,y) #score of every token based on 2nd input token 
print(attn_score)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [12]:
attn_score=inputs @ inputs.T
print(attn_score)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [13]:
attn_weights=torch.softmax(attn_score,dim=1)
print(attn_weights)
print(attn_weights.sum(dim=1))

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [14]:
context_vec=attn_weights @ inputs
context_vec

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

# SELF-ATTENTION WITH TRAINABLE WEIGHTS

In [15]:
x2=inputs[1]
din=inputs.shape[1] #input feature dimension (6, 3)
dout=2 #output (embedding) dimension

In [16]:
torch.manual_seed(123)
w_query=torch.nn.Parameter(torch.rand(din,dout)) #"This tensor is a trainable parameter" 
w_key=torch.nn.Parameter(torch.rand(din,dout)) #torch.rand(din, dout) Creates a matrix of random numbers between 0 and 1
w_value=torch.nn.Parameter(torch.rand(din,dout))

In [17]:
query_2= x2 @ w_query
query_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [18]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [19]:
w_key

Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]], requires_grad=True)

In [20]:
keys= inputs @ w_key
value= inputs @ w_value
print(f"{keys}")

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)


In [21]:
keys_2 = keys[1]
attn_score_22=torch.dot(query_2,keys_2) 

In [22]:
attn_score_22 #This is called attention score (2,2)

tensor(1.8524, grad_fn=<DotBackward0>)

In [23]:
attn_score_2=query_2 @ keys.T
attn_score_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [24]:
d_k=keys.shape[1] #2
attn_weights_2=torch.softmax(attn_score_2/d_k**0.5,dim=-1)
attn_weights_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [25]:
sum(attn_weights_2)

tensor(1., grad_fn=<AddBackward0>)

In [26]:
context_vec_2=attn_weights_2 @ value
context_vec_2

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

# CONTEXT VECTOR FOR ALL INPUTS 

In [27]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, din , dout):
        super().__init__()
        self.w_query=torch.nn.Parameter(torch.rand(din,dout)) #"This tensor is a trainable parameter" 
        self.w_key=torch.nn.Parameter(torch.rand(din,dout)) #torch.rand(din, dout) Creates a matrix of random numbers between 0 and 1
        self.w_value=torch.nn.Parameter(torch.rand(din,dout))

    def forward(self,x):
        queries= inputs @ w_query
        keys= inputs @ w_key
        value= inputs @ w_value

        attn_score=queries @ keys.T
        attn_weights_2=torch.softmax(attn_score/d_k**0.5,dim=-1)
        context_vec=attn_weights @ value
        return context_vec

torch.manual_seed(123)
sa_v1=SelfAttention_v1(din,dout)
sa_v1(inputs)
        

tensor([[0.2897, 0.8043],
        [0.3069, 0.8188],
        [0.3063, 0.8173],
        [0.2972, 0.7936],
        [0.2848, 0.7650],
        [0.3043, 0.8105]], grad_fn=<MmBackward0>)

In [31]:
import torch.nn as nn

class SelfAttention_v2(nn.Module):

    def __init__(self, din, dout, qkv_bias=False):
        super().__init__()
        self.W_query = torch.nn.Linear(din, dout, bias=qkv_bias)
        self.W_key = torch.nn.Linear(din, dout, bias=qkv_bias)
        self.W_value = torch.nn.Linear(din, dout, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)
        context_vec = attn_weights @ values

        return context_vec


torch.manual_seed(789)

sa_v2 = SelfAttention_v2(din, dout)
sa_v2(inputs)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

# ADDING CAUSAL ATTENTION MASK

In [33]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
values = sa_v2.W_value(inputs)

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)

In [34]:
attn_weights

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)

In [35]:
context_length=attn_scores.shape[0]
mask_simple=torch.tril(torch.ones(context_length,context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [36]:
masked_simple=attn_weights *  mask_simple

In [37]:
masked_simple

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)

In [48]:
rows_sums=masked_simple.sum(dim=1,keepdim=True)
print(rows_sums)

tensor([[0.1921],
        [0.3700],
        [0.5357],
        [0.6775],
        [0.8415],
        [1.0000]], grad_fn=<SumBackward1>)


In [49]:
masked_simple_norm=masked_simple / rows_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


## ONE MORE METHOD

In [51]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [52]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


## Masking additional attention weights with dropout

In [53]:
torch.manual_seed(123)
layer=torch.nn.Dropout(0.5)

In [54]:
layer(attn_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)

## Compact Causal SELF-ATTENTION CLASS

In [55]:
batch=torch.stack((inputs,inputs),dim=0)
batch.shape

torch.Size([2, 6, 3])

In [57]:
class CausalAttention(nn.Module):

    def __init__(self, din, dout, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = torch.nn.Linear(din, dout, bias=qkv_bias)
        self.W_key = torch.nn.Linear(din, dout, bias=qkv_bias)
        self.W_value = torch.nn.Linear(din, dout, bias=qkv_bias)
        self.dropout = torch.nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b,num_tokens,din=x.shape # x=batch , 2x6x3
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)  # Changed transpose

        attn_scores.masked_fill_(  # New, _ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens],
            -torch.inf
        )

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,
            dim=-1
        )

        attn_weights = self.dropout(attn_weights)  # New

        context_vec = attn_weights @ values
        return context_vec


torch.manual_seed(789)
dropout=0.0
context_length=batch.shape[1]
ca = CausalAttention(din, dout,context_length,dropout)
ca(batch)

tensor([[[-0.0872,  0.0286],
         [-0.0991,  0.0501],
         [-0.0999,  0.0633],
         [-0.0983,  0.0489],
         [-0.0514,  0.1098],
         [-0.0754,  0.0693]],

        [[-0.0872,  0.0286],
         [-0.0991,  0.0501],
         [-0.0999,  0.0633],
         [-0.0983,  0.0489],
         [-0.0514,  0.1098],
         [-0.0754,  0.0693]]], grad_fn=<UnsafeViewBackward0>)

## Single-head to Multi-head attention

In [58]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads=2, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
                for _ in range(num_heads)
            ]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1]
d_in, d_out = batch.shape[2], 2

mha = MultiHeadAttentionWrapper(
    d_in,
    d_out,
    context_length,
    dropout=0.0,
    num_heads=2
)

mha(batch)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)

## Multi-head attention with Better Efficiency 

In [59]:
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        d_in,
        d_out,
        context_length,
        dropout,
        num_heads,
        qkv_bias=False,
    ):
        super().__init__()

        assert d_out % num_heads == 0, \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        # Projection matrices
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Output projection
        self.out_proj = nn.Linear(d_out, d_out)

        self.dropout = nn.Dropout(dropout)

        # Causal mask
        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1,
            ),
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        # Shape: (b, num_tokens, d_out)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Split into multiple heads
        # (b, num_tokens, d_out)
        # -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(
            b,
            num_tokens,
            self.num_heads,
            self.head_dim,
        )
        queries = queries.view(
            b,
            num_tokens,
            self.num_heads,
            self.head_dim,
        )
        values = values.view(
            b,
            num_tokens,
            self.num_heads,
            self.head_dim,
        )

        # Transpose
        # (b, num_tokens, num_heads, head_dim)
        # -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute attention scores
        attn_scores = queries @ keys.transpose(2, 3)

        # Apply causal mask
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # Scale and normalize
        attn_weights = torch.softmax(
            attn_scores / (keys.shape[-1] ** 0.5),
            dim=-1,
        )
        attn_weights = self.dropout(attn_weights)

        # Compute context vectors
        # (b, num_heads, num_tokens, head_dim)
        context_vec = attn_weights @ values

        # Transpose back
        # (b, num_tokens, num_heads, head_dim)
        context_vec = context_vec.transpose(1, 2)

        # Concatenate heads
        # (b, num_tokens, d_out)
        context_vec = context_vec.contiguous().view(
            b,
            num_tokens,
            self.d_out,
        )

        # Final linear projection
        context_vec = self.out_proj(context_vec)

        return context_vec

In [61]:
torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 4

mha = MultiHeadAttention(
    d_in=d_in,
    d_out=d_out,
    context_length=context_length,
    dropout=0.0,
    num_heads=2,
)

context_vecs = mha(batch)
print(context_vecs)
print(context_vecs.shape)

tensor([[[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]],

        [[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]]], grad_fn=<ViewBackward0>)
torch.Size([2, 6, 4])
